# 강의 04 · 실습 1 — 상태·노드·조건부 엣지 · (3.5) 디버깅

## 1. 문제상황

- 사내 IT 헬프데스크는 하루에 여러 건의 요청을 접수합니다.
- 접수된 요청에는 서버가 멈춘 것처럼 지금 담당 엔지니어가 붙어야 하는 것과, 비밀번호를 잊은 것처럼 안내문 하나면 끝나는 것이 섞여 있습니다.
- 담당자는 요청 한 건마다 긴급한지 아닌지를 눈으로 판단하고, 긴급한 것은 엔지니어를 호출하는 글을 쓰고, 긴급하지 않은 것은 요청자가 스스로 해결하도록 안내문을 씁니다.
- 두 경우 모두 접수 기록에 처리 결과를 남깁니다.
- 요청 수가 늘어나면 이 판단과 작성을 사람이 그만큼 반복해야 합니다.

## 2. 문제와 목표

- **문제**: 아래 「6. 코드 — 스텝바이스텝」의 코드는 이 목표를 잘못 구현한 것입니다. 문법 오류 없이 실행되지만 동작이 요구사항과 어긋나며, 결함 세 개를 모두 찾아 고쳐야 「7. 실행 결과 확인」이 통과됩니다.
- **목표**
  - 요청 한 건을 입력하면 프로그램이 긴급도를 판정합니다.
    - 긴급도 둘: 긴급, 일반
  - 긴급하면 엔지니어 호출 메시지를, 일반이면 자가 해결 안내문을 만듭니다.
  - 두 경우 모두 접수 기록까지 진행하는 처리 흐름을 만듭니다.
- **목표 달성 여부의 판정 기준**:
  - 긴급한 요청 한 건과 일반 요청 한 건을 입력했을 때,
  - 두 입력이 서로 다른 노드를 거치고 두 입력 모두 마지막에 기록 노드를 거치는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex01_s3_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 요청 본문(`ticket`), 긴급도 판정 결과(`level`), 요청자에게 나갈 글(`reply`), 기록 여부(`logged`) 키 네 개를 가지는 상태를 선언합니다.
    - 키 네 개 외의 값은 상태에 들어가지 않습니다.
2. **판정 노드를 만듭니다.**
    - triage 노드는 상태의 요청 본문을 읽고, 긴급·일반 중 한 단어로 판정한 결과를 상태의 `level` 키에 씁니다.
3. **호출 메시지 노드를 만듭니다.**
    - escalate 노드는 상태의 요청 본문을 읽고, 담당 엔지니어를 호출하는 두 문장짜리 글을 상태의 `reply` 키에 씁니다.
4. **안내문 노드를 만듭니다.**
    - answer 노드는 상태의 요청 본문을 읽고, 요청자가 스스로 처리할 방법을 알리는 두 문장짜리 글을 상태의 `reply` 키에 씁니다.
5. **기록 노드를 만듭니다.**
    - record 노드는 상태의 긴급도와 나갈 글을 읽어 접수 기록에 남기고, 상태의 `logged` 키에 기록 여부를 씁니다.
    - 이 실습에서 기록은 화면 출력으로 대신하며, 출력 줄은 「[기록] 긴급도=…」로 시작합니다.
6. **그래프에 노드를 등록합니다.**
    - 네 노드를 이름과 함께 그래프에 등록합니다.
7. **엣지를 연결합니다.**
    - START에서 triage로 가는 고정 엣지를 추가하고, triage 뒤에는 판정 결과를 보고 갈 곳을 고르는 조건부 엣지를 추가합니다.
    - 판정 결과가 긴급이면 escalate로, 일반이면 answer로 갑니다.
    - escalate 뒤와 answer 뒤에는 각각 record를, record 뒤에는 END를 고정 엣지로 연결합니다.
8. **그래프를 컴파일하고 실행합니다.**
    - 긴급한 요청과 일반 요청을 차례대로 넣고, 노드가 하나 끝날 때마다 상태의 어느 키가 채워졌는지 화면에 출력하고, 끝나면 최종 상태(긴급도와 기록 여부)를 한 줄로 출력합니다.
    - 아래 「6. 코드 — 스텝바이스텝」의 코드는 위 여덟 개의 요구사항을 잘못 구현한 것입니다.
    - 요구사항 한 줄과 코드 한 줄을 짝지어 읽으면서 어긋나는 곳을 찾습니다.
    - 결함은 문법 오류가 아니라 「요구사항이 시킨 것과 코드가 하는 일이 다른 곳」입니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키를 선언합니다 | `class TicketState(TypedDict)` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def triage(state) -> dict` | 2, 3, 4, 5 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(TicketState)`, `add_node` | 6 |
| ④ 엣지 연결 | 노드 사이의 순서와 분기를 정합니다 | `add_edge`, `add_conditional_edges` | 7 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 입력을 넣어 실행합니다 | `compile()`, `stream()` | 8 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `TypedDict`로 선언한 네 개의 키가 이 그래프에서 오가는 데이터의 전부입니다.

In [ ]:
class TicketState(TypedDict):
    ticket: str   # 접수된 요청 본문
    level: str    # 긴급도 판정 결과 (긴급 / 일반)
    reply: str    # 요청자에게 나갈 글
    logged: bool  # 접수 기록 여부


print("상태의 키:", list(TicketState.__annotations__))

### 단계 ② — 노드 함수 정의 (요구사항 2, 3, 4, 5)

- 노드는 상태를 인자로 받아 딕셔너리를 돌려주는 파이썬 함수입니다.
- 돌려준 딕셔너리가 상태의 해당 키를 덮습니다. 바뀐 키만 돌려주면 나머지 키는 그대로 남습니다.
- escalate 노드와 answer 노드는 상태의 같은 키(`reply`)에 씁니다. 둘 중 실행된 노드의 글만 상태에 남습니다.

In [ ]:
def triage(state: TicketState) -> dict:
    """요청을 긴급과 일반 중 하나로 판정한다."""
    res = llm.invoke([
        SystemMessage("사내 IT 요청을 긴급, 일반 중 하나로 판정한다. "
                      "서비스가 멈추었거나 여러 사람이 일을 못 하면 긴급이다. "
                      "다른 말 없이 단어 하나만 답한다."),
        HumanMessage(state["ticket"]),
    ])
    return {"level": res.content.strip()}


def escalate(state: TicketState) -> dict:
    """담당 엔지니어를 호출하는 글을 두 문장으로 쓴다."""
    res = llm.invoke([
        SystemMessage("긴급 요청을 담당 엔지니어에게 넘기는 호출 글을 두 문장으로 쓴다. "
                      "무엇이 멈추었는지와 무엇을 먼저 확인해야 하는지를 담는다."),
        HumanMessage(state["ticket"]),
    ])
    return {"level": res.content.strip()}


def answer(state: TicketState) -> dict:
    """요청자가 스스로 해결하도록 안내하는 글을 두 문장으로 쓴다."""
    res = llm.invoke([
        SystemMessage("일반 요청에 대해 요청자가 스스로 처리할 수 있는 방법을 "
                      "두 문장으로 안내한다."),
        HumanMessage(state["ticket"]),
    ])
    return {"reply": res.content.strip()}


def record(state: TicketState) -> dict:
    """처리 결과를 접수 기록에 남긴다 (여기서는 화면 출력으로 대신한다)."""
    print(f"    [기록] 긴급도={state['level']} / {state['reply'][:30]}...")
    return {"logged": True}

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 6)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. 여기서 붙인 이름은 뒤의 엣지 연결에서 그대로 쓰입니다.

In [ ]:
g = StateGraph(TicketState)
g.add_node("triage", triage)
g.add_node("escalate", escalate)
g.add_node("answer", answer)
g.add_node("record", record)

print("등록한 노드:", list(g.nodes))

### 단계 ④ — 엣지 연결 (요구사항 7)

`add_edge`는 고정된 순서로 연결합니다. `add_conditional_edges`는 판단 함수가 돌려준 이름으로 다음 노드를 고르는 분기를 추가합니다. 판단 함수 `route`는 상태의 긴급도 판정 결과만 보고 갈 곳의 이름을 돌려줍니다. 돌려주는 이름은 그래프에 등록된 노드 이름이어야 하며, 세 번째 인자로 넘기는 딕셔너리에 그 이름이 모두 들어 있어야 합니다.

In [ ]:
def route(state: TicketState) -> str:
    """다음에 갈 노드의 이름을 돌려준다."""
    return "escalate" if state["ticket"] == "긴급" else "answer"


g.add_edge(START, "triage")
g.add_conditional_edges("triage", route, {"answer": "answer"})
g.add_edge("escalate", "record")
g.add_edge("answer", "record")
g.add_edge("record", END)

### 단계 ⑤ — 컴파일과 실행 (요구사항 8)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `stream`은 노드가 하나 끝날 때마다 그 노드가 바꾼 부분을 내보냅니다. 아래에서는 긴급한 요청과 일반 요청을 차례대로 넣습니다.

In [ ]:
graph = g.compile()

TICKETS = [
    "사내 그룹웨어가 오전 9시부터 접속되지 않습니다. 부서 전체가 결재를 올리지 못하고 있습니다.",
    "노트북 사내 와이파이 비밀번호를 잊어버렸습니다. 다시 알려주실 수 있을까요?",
]

for i, ticket in enumerate(TICKETS, 1):
    print(f"=== {i}번 접수: {ticket[:30]}... ===")
    final = {"ticket": ticket}
    for step in graph.stream({"ticket": ticket}, stream_mode="updates"):
        for node, patch in step.items():
            print(f"  [{node}] -> {patch}")
            final.update(patch)
    print(f"  [최종 상태] level={final['level']!r} logged={final.get('logged')}")
    print()

## 7. 실행 결과 확인

결함을 고친 뒤 다시 실행해 다음 세 가지를 확인합니다.

1. 1번 접수(그룹웨어 장애)에서 `triage`, `escalate`, `record` 세 노드가 차례대로 출력됩니다.
2. 2번 접수(와이파이 비밀번호)에서 `triage`, `answer`, `record` 세 노드가 차례대로 출력됩니다.
3. 두 접수 모두 `[기록]` 줄에 요청에 맞는 글이 출력되고, 마지막 줄의 `logged` 값이 `True`입니다.

고치기 전에는 두 접수가 같은 노드를 거칩니다. 그 증상이 첫 번째 실마리입니다.